# Zero-shot Qwen2.5-VL-3B-Instruct baseline

Runs the **base** model — no LoRA adapter, no fine-tuning — over the exact
same test set used for the fine-tuned model, so results are directly
comparable row-for-row. This is the control your mentor flagged in point 8:
without it, there is no evidence the fine-tuned model\'s results actually came
from fine-tuning rather than the base model already handling this domain.

Output is saved in the same schema as `visionllm_predictions.csv`
(`image_full_path`, `question`, `ground_truth`, `task`, `prediction`) so it can
be dropped directly into the same metrics notebook used for the fine-tuned
model, just pointed at this file instead.

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 66.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import json
import pandas as pd
from pathlib import Path

with open("/content/drive/MyDrive/Surgical-VLM/configs/config.json") as f:
    CONFIG = json.load(f)

PROCESSED_DIR = Path(CONFIG["processed_dir"])  # adjust key name if different in your config.json
print("PROCESSED_DIR:", PROCESSED_DIR)


PROCESSED_DIR: /content/drive/MyDrive/Surgical-VLM/processed


In [ ]:
eval_df = pd.read_parquet(PROCESSED_DIR / "matched_test.parquet")
print(eval_df.shape)
print(eval_df["task"].value_counts())
eval_df.head()


(5600, 18)
task
Action Recognition              800
Instrument Recognition          800
Phase Recognition               800
Safety Assessment               800
Surgical Image Captioning       800
Tissue and Organ Recognition    800
Triplet Recognition             800
Name: count, dtype: int64


,sample_id,image_id,case_id,image_path,image_full_path,image_exists,source_dataset,specialty,surgery_type,question,thinking,answer,task,task_group,gt_label,annotator,text,label_set
0,517834,29615,VID110,CholecT50/videos/VID110/001493.png,/content/CholecT50/CholecT50/videos/VID110/001...,True,CholecT50,Hepatobiliary,Cholecystectomy,Identify the grasper's action in this surgery ...,None,The grasper is performing a retract action in ...,Action Recognition,Understanding and Reasoning,"[{""actions"": [""retract""]}]",NUS,The grasper is performing a retract action in ...,[]
1,32083,67707,VID68,CholecT50/videos/VID68/001508.png,/content/CholecT50/CholecT50/videos/VID68/0015...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,"retract, dissect",Action Recognition,Understanding and Reasoning,"[""retract, dissect""]",SJTU,"retract, dissect",[]
2,552118,31976,VID36,CholecT50/videos/VID36/000224.png,/content/CholecT50/CholecT50/videos/VID36/0002...,True,CholecT50,Hepatobiliary,Cholecystectomy,What is the grasper doing in this surgical scene?,None,The grasper is performing a retract action in ...,Action Recognition,Understanding and Reasoning,"[{""actions"": [""retract""]}]",NUS,The grasper is performing a retract action in ...,[]
3,33687,29387,VID05,CholecT50/videos/VID05/000293.png,/content/CholecT50/CholecT50/videos/VID05/0002...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,"retract, dissect",Action Recognition,Understanding and Reasoning,"[""retract, dissect""]",SJTU,"retract, dissect",[]
4,41279,31695,VID36,CholecT50/videos/VID36/001288.png,/content/CholecT50/CholecT50/videos/VID36/0012...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,dissect,Action Recognition,Understanding and Reasoning,"[""dissect""]",SJTU,dissect,[]


## Load the base model — no adapter

Unquantized bf16, same reasoning as the fine-tuned eval: no LoRA to merge here
since this is the base model being tested as-is, and unquantized is faster for
inference than 4-bit (no dequantization overhead per forward pass).

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

processor.tokenizer.padding_side = "left"  # required for correct batched generation

print("Model vocab / embedding check:")
print("Processor vocab size:", len(processor.tokenizer))
print("Model embedding rows:", model.get_input_embeddings().weight.shape[0])


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model vocab / embedding check:
Processor vocab size: 151665
Model embedding rows: 151936


In [ ]:
import time, shutil, subprocess
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

DRIVE_CHOLECT50_ARCHIVE = Path("/content/drive/MyDrive/Surgical-VLM/data/CholecT50_raw.zip")  # adjust extension if .tar.gz
LOCAL_ARCHIVE_COPY = Path("/content/CholecT50_raw" + DRIVE_CHOLECT50_ARCHIVE.suffix)
LOCAL_CHOLECT50_DIR = Path("/content/CholecT50")

assert DRIVE_CHOLECT50_ARCHIVE.exists(), (
    f"Expected archive not found at {DRIVE_CHOLECT50_ARCHIVE}. "
    "Upload it to Drive first (see markdown above) before running this cell."
)


print("Copying archive to local disk..")
t0 = time.time()
shutil.copy2(DRIVE_CHOLECT50_ARCHIVE, LOCAL_ARCHIVE_COPY)
print(f"Copy done in {time.time()-t0:.1f}s")

LOCAL_CHOLECT50_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting locally...")
t0 = time.time()
if LOCAL_ARCHIVE_COPY.suffix == ".zip":
    subprocess.run(["unzip", "-q", str(LOCAL_ARCHIVE_COPY), "-d", str(LOCAL_CHOLECT50_DIR)], check=True)
else:
    subprocess.run(["tar", "-xzf", str(LOCAL_ARCHIVE_COPY), "-C", str(LOCAL_CHOLECT50_DIR)], check=True)
print(f"Extract done in {time.time()-t0:.1f}s")

n_files = sum(1 for _ in LOCAL_CHOLECT50_DIR.rglob("*") if _.is_file())
print(f"\n{n_files:,} files extracted to {LOCAL_CHOLECT50_DIR}")

# Free the local zip copy now that it's extracted -- no need to keep both
LOCAL_ARCHIVE_COPY.unlink()



Copying archive to local disk..
Copy done in 1195.3s
Extracting locally...
Extract done in 634.7s

100,918 files extracted to /content/CholecT50


## Sanity check on one sample before the full run


In [ ]:
from PIL import Image

sample = eval_df.iloc[0]
image = Image.open(sample["image_full_path"]).convert("RGB")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": sample["question"]},
    ],
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

new_tokens = generated_ids[:, input_len:]
answer = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

print("Question:", sample["question"])
print("Ground Truth:", sample["text"])
print("Zero-shot Prediction:", answer)


Question: Identify the grasper's action in this surgery image.
Ground Truth: The grasper is performing a retract action in this surgery image.
Zero-shot Prediction: In the provided image, the grasper is being used to grasp and manipulate tissue within the surgical field. The grasper appears to be positioned near the edge of a wound or incision, suggesting that it might be being used to either stabilize tissue for suturing or to perform some form of manipulation during the procedure. The


## Batched generation over the full test set

In [ ]:
from tqdm.auto import tqdm

RESULTS_PATH = "/content/drive/MyDrive/Surgical-VLM/results/visionllm_zeroshot_predictions.csv"

def generate_batch_zeroshot(rows, batch_size=8):
    predictions = []
    for i in tqdm(range(0, len(rows), batch_size)):
        batch_rows = rows.iloc[i:i+batch_size]
        try:
            images = [Image.open(p).convert("RGB") for p in batch_rows.image_full_path]
            texts = []
            for img, q in zip(images, batch_rows.question):
                messages = [{"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": q},
                ]}]
                texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

            inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
            input_len = inputs["input_ids"].shape[1]

            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

            new_tokens = generated_ids[:, input_len:]
            batch_preds = processor.batch_decode(new_tokens, skip_special_tokens=True)

        except Exception as e:
            print(f"Batch {i}-{i+batch_size} failed: {e}")
            batch_preds = [""] * len(batch_rows)

        predictions.extend(batch_preds)

        # checkpoint partial results every batch, same schema as visionllm_predictions.csv
        partial = rows.iloc[:len(predictions)].copy()
        partial["prediction"] = predictions
        partial[["image_full_path", "question", "text", "task", "prediction"]].to_csv(
            RESULTS_PATH, index=False
        )

    return predictions

predictions = generate_batch_zeroshot(eval_df, batch_size=8)


  0%|          | 0/700 [00:00<?, ?it/s]

In [ ]:
zeroshot_results = eval_df.copy()
zeroshot_results["prediction"] = predictions
zeroshot_results = zeroshot_results[["image_full_path", "question", "text", "task", "prediction"]]

zeroshot_results.to_csv(RESULTS_PATH, index=False)
print(f"Saved {len(zeroshot_results)} zero-shot predictions to {RESULTS_PATH}")
zeroshot_results.head()


Saved 5600 zero-shot predictions to /content/drive/MyDrive/Surgical-VLM/results/visionllm_zeroshot_predictions.csv


,image_full_path,question,text,task,prediction
0,/content/CholecT50/CholecT50/videos/VID110/001...,Identify the grasper's action in this surgery ...,The grasper is performing a retract action in ...,Action Recognition,"In the provided image, the grasper is being us..."
1,/content/CholecT50/CholecT50/videos/VID68/0015...,"Given the laparoscopic cholecystectomy image, ...","retract, dissect",Action Recognition,In the provided image from a laparoscopic chol...
2,/content/CholecT50/CholecT50/videos/VID36/0002...,What is the grasper doing in this surgical scene?,The grasper is performing a retract action in ...,Action Recognition,"In this surgical scene, the grasper is being u..."
3,/content/CholecT50/CholecT50/videos/VID05/0002...,"Given the laparoscopic cholecystectomy image, ...","retract, dissect",Action Recognition,In the image from a laparoscopic cholecystecto...
4,/content/CholecT50/CholecT50/videos/VID36/0012...,"Given the laparoscopic cholecystectomy image, ...",dissect,Action Recognition,In the provided image from a laparoscopic chol...
